In [1]:
import pandas as pd
import numpy as np
import s3fs
from datetime import datetime

# MinIO настройки
MINIO_ENDPOINT = "minio:9000"
ACCESS_KEY = "minioadmin"
SECRET_KEY = "minioadmin"
BUCKET = "oil-data"

fs = s3fs.S3FileSystem(
    key=ACCESS_KEY,
    secret=SECRET_KEY,
    client_kwargs={'endpoint_url': f'http://{MINIO_ENDPOINT}'}
)

# Функция для чтения всех партиций таблицы
def read_partitioned_table(table_name):
    """Читает все parquet файлы из s3://oil-data/table_name/**/*.parquet"""
    pattern = f"{BUCKET}/{table_name}/**/*.parquet"
    files = fs.glob(pattern)
    if not files:
        raise ValueError(f"No files found for {table_name}")
    dfs = []
    for f in files:
        with fs.open(f, 'rb') as fp:
            dfs.append(pd.read_parquet(fp))
    return pd.concat(dfs, ignore_index=True)

# Читаем таблицы
print("Reading production...")
df_prod = read_partitioned_table("production")
print(f"Raw production rows: {len(df_prod)}")

print("Reading wells...")
df_wells = read_partitioned_table("wells")
print(f"Wells: {len(df_wells)}")

Reading production...
Raw production rows: 150
Reading wells...
Wells: 5


In [2]:
# Обработка NULL в production

# Проверим NULL значения
print("NULL values before cleaning:")
print(df_prod.isnull().sum())

# Для температуры и давления: заполняем медианой по скважине (активные скважины не имеют NULL, но для suspended заполним)
df_prod['temperature'] = df_prod.groupby('well_id')['temperature'].transform(lambda x: x.fillna(x.median()))
df_prod['pressure'] = df_prod.groupby('well_id')['pressure'].transform(lambda x: x.fillna(x.median()))

# Если остались NULL (например, у скважины 4 все данные NULL) – заменим на 0 или среднее по полю
df_prod['temperature'].fillna(df_prod['temperature'].mean(), inplace=True)
df_prod['pressure'].fillna(df_prod['pressure'].mean(), inplace=True)
df_prod['oil_ton'].fillna(0, inplace=True)
df_prod['gas_m3'].fillna(0, inplace=True)
df_prod['water_m3'].fillna(0, inplace=True)
df_prod['energy_kwh'].fillna(0, inplace=True)
df_prod['downtime_hours'].fillna(0, inplace=True)

print("NULL values after cleaning:")
print(df_prod.isnull().sum())

NULL values before cleaning:
prod_id            0
well_id            0
date               0
oil_ton            0
gas_m3             0
water_m3           0
energy_kwh         0
downtime_hours     0
temperature       30
pressure          30
year               0
month              0
day                0
dtype: int64
NULL values after cleaning:
prod_id           0
well_id           0
date              0
oil_ton           0
gas_m3            0
water_m3          0
energy_kwh        0
downtime_hours    0
temperature       0
pressure          0
year              0
month             0
day               0
dtype: int64


In [3]:
# Фильтрация выбросов для oil_ton

from scipy import stats

z_scores = np.abs(stats.zscore(df_prod['oil_ton'].fillna(0)))
df_prod_clean = df_prod[z_scores < 3]
print(f"Removed {len(df_prod) - len(df_prod_clean)} outliers from production")
df_prod = df_prod_clean

Removed 0 outliers from production


In [4]:
# Создание агрегаций
# Общая добыча по дням

daily_production = df_prod.groupby('date').agg(
    total_oil_ton=('oil_ton', 'sum'),
    total_gas_m3=('gas_m3', 'sum'),
    total_water_m3=('water_m3', 'sum'),
    total_energy_kwh=('energy_kwh', 'sum'),
    avg_temperature=('temperature', 'mean'),
    avg_pressure=('pressure', 'mean')
).reset_index()

print("Daily production sample:")
print(daily_production.head())

Daily production sample:
        date  total_oil_ton  total_gas_m3  total_water_m3  total_energy_kwh  \
0 2025-10-01          717.5      197170.0           631.1           26730.0   
1 2025-10-02          717.3      197200.0           630.5           26735.0   
2 2025-10-03          719.2      197350.0           630.8           26770.0   
3 2025-10-04          722.9      197810.0           627.4           26875.0   
4 2025-10-05          721.0      197620.0           630.1           26810.0   

   avg_temperature  avg_pressure  
0        84.646333    115.461833  
1        84.646333    115.501833  
2        84.846333    115.521833  
3        84.446333    115.921833  
4        84.686333    115.701833  


In [5]:
# KPI по скважинам (средний дебит, % простоя, среднее давление, средняя температура, коэффициент простоя)

# Коэффициент простоя (доля времени простоя за день)
df_prod['downtime_ratio'] = df_prod['downtime_hours'] / 24.0

well_kpi = df_prod.groupby('well_id').agg(
    avg_oil_ton=('oil_ton', 'mean'),
    avg_downtime_ratio=('downtime_ratio', 'mean'),
    avg_temperature=('temperature', 'mean'),
    avg_pressure=('pressure', 'mean'),
    total_downtime_hours=('downtime_hours', 'sum'),
    days_active=('date', 'nunique')
).reset_index()

# Добавим название скважины
well_kpi = well_kpi.merge(df_wells[['well_id', 'name', 'field_name', 'region', 'status']], on='well_id', how='left')

print("Well KPIs:")
print(well_kpi.head())

Well KPIs:
   well_id  avg_oil_ton  avg_downtime_ratio  avg_temperature  avg_pressure  \
0        1   213.150000            0.020139        88.206667    120.543333   
1        2   185.816667            0.030278        84.486667    115.306667   
2        3   121.696667            0.081111        79.433333    107.160000   
3        4     0.000000            1.000000        84.731667    115.609167   
4        5   198.430000            0.018194        86.800000    119.426667   

   total_downtime_hours  days_active      name    field_name         region  \
0                  14.5           30  Well-101   Severo-Ural   Khanty-Mansi   
1                  21.8           30  Well-102   Severo-Ural   Khanty-Mansi   
2                  58.4           30  Well-203  Vostok-Field  Yamalo-Nenets   
3                 720.0           30  Well-304   Zapad-Field          Tomsk   
4                  13.1           30  Well-305   Zapad-Field          Tomsk   

        status  
0       active  
1       act

In [6]:
# Сохранение очищенных данных и витрин в MinIO

# Функция для сохранения DataFrame в MinIO
def save_to_minio(df, path):
    with fs.open(f"{BUCKET}/{path}", 'wb') as f:
        df.to_parquet(f, index=False)
    print(f"Saved to {BUCKET}/{path}")

# Сохраняем очищенную таблицу production
save_to_minio(df_prod, "cleaned/production.parquet")

# Сохраняем витрины
save_to_minio(daily_production, "marts/daily_production.parquet")
save_to_minio(well_kpi, "marts/well_kpi.parquet")

Saved to oil-data/cleaned/production.parquet
Saved to oil-data/marts/daily_production.parquet
Saved to oil-data/marts/well_kpi.parquet


In [7]:
# Создаём фичи для ML (агрегация well_telemetry по дням)

# Читаем well_telemetry
print("Reading well_telemetry...")
df_tele = read_partitioned_table("well_telemetry")
print(f"Telemetry rows: {len(df_tele)}")

# Создаём колонку date из timestamp
df_tele['date'] = pd.to_datetime(df_tele['timestamp']).dt.date

# Агрегируем по well_id и date
tele_daily = df_tele.groupby(['well_id', 'date']).agg(
    avg_pump_speed=('pump_speed_rpm', 'mean'),
    avg_pump_current=('pump_current', 'mean'),
    avg_pressure_in=('pressure_in', 'mean'),
    avg_pressure_out=('pressure_out', 'mean'),
    avg_temperature=('temperature', 'mean'),
    avg_vibration=('vibration', 'mean'),
    avg_oil_flow_rate=('oil_flow_rate', 'mean')
).reset_index()

# Читаем well_targets
df_targets = read_partitioned_table("well_targets")
# Приводим date к тому же типу
df_targets['date'] = pd.to_datetime(df_targets['date']).dt.date

# Объединяем с targets
ml_dataset = tele_daily.merge(df_targets, on=['well_id', 'date'], how='inner')
print(f"ML dataset shape: {ml_dataset.shape}")
save_to_minio(ml_dataset, "marts/ml_dataset.parquet")

Reading well_telemetry...
Telemetry rows: 48
ML dataset shape: (2, 10)
Saved to oil-data/marts/ml_dataset.parquet


In [8]:
# Проверка сохранённых файлов

for path in ["cleaned/production.parquet", "marts/daily_production.parquet", "marts/well_kpi.parquet", "marts/ml_dataset.parquet"]:
    if fs.exists(f"{BUCKET}/{path}"):
        print(f"+ {path}")
    else:
        print(f"- {path} missing")

+ cleaned/production.parquet
+ marts/daily_production.parquet
+ marts/well_kpi.parquet
+ marts/ml_dataset.parquet
